# Trainable Mixture-of-Experts

**Audience:** ML practitioners who know Python and basic linear algebra.  
**Prerequisites:** NumPy, optimization basics, and the project README.  
**Learning goals:** implement the core algorithm, validate invariants, and evaluate a deterministic end-to-end run.

## Outline
1. Configure the reference system.
2. Execute its core training or simulation loop.
3. Verify numerical and behavioral assertions.
4. Try the exercise and inspect pitfalls.


In [1]:
from pathlib import Path
import sys, numpy as np
ROOT=Path.cwd()
if not (ROOT/'src').exists():
    ROOT=Path('/Users/shivamkumar/Documents/Codex/ml/implementations/trainable-mixture-of-experts-in-cuda')
sys.path.insert(0,str(ROOT/'src'))
np.set_printoptions(precision=3,suppress=True)


## 1. Route tokens to top-k experts
This NumPy oracle validates semantics before a CUDA dispatch kernel is optimized.


In [2]:
from moe.core import *
cfg=MoEConfig(steps=35)
p=init(cfg); rng=np.random.default_rng(0); x=rng.normal(size=(cfg.tokens,cfg.d_model))
y,meta=forward(p,x,cfg)
print('counts:',meta['counts'],'capacity:',meta['capacity'],'aux:',round(meta['aux_loss'],3))


counts: [35 27 30 36] capacity: 48 aux: 1.0


## 2. Train expert weights and evaluate
The small loop exercises route, dispatch, combine, and expert backward paths.


In [3]:
out=train(cfg)
print('loss:',round(out['history'][0],4),'→',round(out['loss'],4))


loss: 0.4827 → 0.0709


In [4]:
assert out['loss']<out['history'][0]*.75
assert np.all(out['metadata']['counts']<=out['metadata']['capacity'])
print('MoE reference passed')


MoE reference passed


## Exercise
Change one capacity, rank, learning-rate, sample-size, or risk parameter in `config.json`. Predict the direction of the primary metric before rerunning.


In [5]:
# Answer scaffold: record the parameter, prediction, and observed metric.
exercise = {'parameter': 'choose one', 'prediction': 'state a direction', 'observed': None}
print(exercise)


{'parameter': 'choose one', 'prediction': 'state a direction', 'observed': None}


## Pitfalls and extensions
**Pitfall:** comparing runs with different seeds or compute budgets can masquerade as an algorithmic gain. Keep the seed and budget fixed.  
**Extension:** add a larger held-out scenario and report both quality and resource cost.
